# Differentiable feature extraction with `AFX.torch`

`AFX.torch.functional` reimplements the STFT, mel spectrogram, MFCC and spectral descriptors in torch with the same numbers as the NumPy path. The functions take `(time,)` or `(batch, time)` tensors on CPU or CUDA, and gradients flow back to the waveform. `AFXFeatureModule` wraps them as an `nn.Module` front end.

Requires `pip install "afx-audio[torch]"`.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import torch

from AFX.io.io import load_audio
from AFX.extractors import cepstral_features, frequency_domain
from AFX.torch import AFXFeatureModule, functional as F_afx, supported_features

device = 'cuda' if torch.cuda.is_available() else 'cpu'
SR = 22050
print(f'torch {torch.__version__}, device = {device}')
print('Features available on the torch path:', supported_features())

## Load audio

We use a clip from an environmental-sound dataset (Nonspeech7k, ESC-50, or UrbanSound8K) resolved via environment variables, falling back to a synthesized clip so the notebook runs anywhere.

In [ ]:
import glob

# Environmental-sound dataset roots are passed via environment variables.
# Set any ONE of these before launching Jupyter, e.g.
#   export ESC_50_dataset=/path/to/ESC-50
DATASET_ENV_VARS = ['dataset_nonspeech7k', 'nonspeech7k_dataset',
                    'ESC_50_dataset', 'UrbanSound8K_dataset']

def find_dataset_audio(max_files=None, exts=('.wav', '.ogg', '.flac')):
    """Return audio file paths from the first configured dataset env var
    (empty list if no dataset is configured)."""
    for var in DATASET_ENV_VARS:
        root = os.environ.get(var)
        if root and os.path.isdir(root):
            files = sorted(
                p for ext in exts for p in
                glob.glob(os.path.join(root, '**', '*' + ext), recursive=True)
            )
            if files:
                print(f'Dataset: {var} -> {root} ({len(files)} audio files)')
            return files[:max_files] if max_files else files
    return []

def synth_clip(kind, seconds=1.0, sr=22050, seed=0):
    """Synthesize a simple environmental-sound-like clip (fallback when no
    dataset env var is configured)."""
    rng = np.random.default_rng(seed)
    t = np.arange(int(sr * seconds)) / sr
    if kind == 'tonal':        # bird-chirp-like rising tone
        f = 800 + 600 * t
        sig = 0.6 * np.sin(2 * np.pi * f * t)
    elif kind == 'noisy':      # rain-like broadband noise
        sig = 0.4 * rng.standard_normal(t.size)
    else:                      # engine-like low harmonic stack
        sig = sum(0.3 / k * np.sin(2 * np.pi * 90 * k * t) for k in (1, 2, 3))
    sig = sig + 0.02 * rng.standard_normal(t.size)
    return sig.astype(np.float32)

files = find_dataset_audio(max_files=4)
if files:
    signal = load_audio(files[0], sr=SR)[0][:SR]  # first second
else:
    print('No dataset env var set - using synthetic audio.')
    signal = synth_clip('tonal', sr=SR)
waveform = torch.as_tensor(signal, dtype=torch.float32, device=device)
print('waveform:', waveform.shape, waveform.dtype, waveform.device)

## Functional API: parity with the NumPy path

The same MFCCs from the NumPy extractor and from the torch function.

In [ ]:
mfcc_np = cepstral_features.extract_mfcc(
    np.asarray(signal, dtype=np.float64), SR,
    n_mfcc=13, frame_size=1024, hop_length=256)['mfcc']
mfcc_pt = F_afx.mfcc(waveform.double(), SR, n_mfcc=13,
                     frame_size=1024, hop_length=256).cpu().numpy()

print('shapes:', mfcc_np.shape, mfcc_pt.shape)
print('max abs difference:', np.abs(mfcc_np - mfcc_pt).max())

fig, ax = plt.subplots(1, 3, figsize=(14, 3.2))
for a, m, title in zip(ax, [mfcc_np, mfcc_pt, np.abs(mfcc_np - mfcc_pt)],
                       ['NumPy path', 'torch path', '|difference|']):
    im = a.imshow(m, aspect='auto', origin='lower', cmap='magma')
    a.set_title(title); a.set_xlabel('frame'); a.set_ylabel('MFCC')
    fig.colorbar(im, ax=a)
plt.tight_layout()

## Batched extraction

Several clips stacked into a `(batch, time)` tensor; every functional transform broadcasts over the batch axis, which is where a GPU helps.

In [ ]:
if files and len(files) >= 3:
    clips = [load_audio(f, sr=SR)[0][:SR] for f in files[:3]]
    clips = [np.pad(c, (0, SR - len(c))) for c in clips]  # equal length
else:
    clips = [synth_clip(k, sr=SR, seed=i)
             for i, k in enumerate(['tonal', 'noisy', 'harmonic'])]
batch = torch.as_tensor(np.stack(clips), dtype=torch.float32, device=device)

mel_batch = F_afx.melspectrogram(batch, SR, n_mels=64,
                                 frame_size=1024, hop_length=256)
cent_batch = F_afx.spectral_centroid(batch, SR, frame_size=1024,
                                     hop_length=256)
print('batch:', batch.shape)
print('mel spectrograms:', mel_batch.shape)   # (batch, n_mels, frames)
print('spectral centroid:', cent_batch.shape) # (batch, frames)

fig, ax = plt.subplots(1, 3, figsize=(14, 3))
for i in range(3):
    ax[i].imshow(10 * np.log10(mel_batch[i].cpu().numpy() + 1e-10),
                 aspect='auto', origin='lower', cmap='viridis')
    ax[i].set_title(f'clip {i} (log-mel)')
plt.tight_layout()

## Gradients flow through the features

The transforms are differentiable, so you can optimize a *waveform* against a feature-space loss. Here we start from noise and gradient-descend until its MFCCs match the target clip's MFCCs.

In [ ]:
target_mfcc = F_afx.mfcc(waveform, SR, n_mfcc=13,
                         frame_size=1024, hop_length=256).detach()

x = torch.randn_like(waveform) * 0.01
x.requires_grad_(True)
opt = torch.optim.Adam([x], lr=1e-2)

losses = []
for step in range(60):
    opt.zero_grad()
    loss = torch.nn.functional.mse_loss(
        F_afx.mfcc(x, SR, n_mfcc=13, frame_size=1024, hop_length=256),
        target_mfcc)
    loss.backward()
    opt.step()
    losses.append(loss.item())

plt.figure(figsize=(6, 3))
plt.semilogy(losses)
plt.xlabel('step'); plt.ylabel('MFCC MSE (log scale)')
plt.title(f'Waveform optimization: {losses[0]:.1f} -> {losses[-1]:.3f}')
plt.tight_layout()

## `AFXFeatureModule`: a feature front end for models

The module mirrors the `extract_all_features` config convention. Constants (windows, mel filterbanks, DCT matrices) are registered as buffers, so `.to(device)` moves everything. With `output='vector'` it plugs straight into `nn.Sequential`.

In [ ]:
frontend = AFXFeatureModule(
    features={
        'mfcc': {'n_mfcc': 13, 'frame_size': 1024, 'hop_length': 256},
        'spectral_centroid': {'frame_size': 1024, 'hop_length': 256},
        'spectral_flatness': {'frame_size': 1024, 'hop_length': 256},
    },
    sample_rate=SR, aggregation='mean', output='vector',
).to(device)

vec = frontend(batch)
print('feature vectors:', vec.shape)  # (batch, 13 + 1 + 1)
print(frontend)

### Training a small classifier through the front end

Two synthetic classes (tonal and noisy clips). The classifier trains end to end through the feature extraction, so gradients reach the waveforms, which is what trainable preprocessing or feature-space losses rely on.

In [ ]:
torch.manual_seed(0)
n_per_class = 8
X = np.stack([synth_clip('tonal', sr=SR, seed=i) for i in range(n_per_class)]
             + [synth_clip('noisy', sr=SR, seed=i) for i in range(n_per_class)])
y = torch.tensor([0] * n_per_class + [1] * n_per_class, device=device)
Xt = torch.as_tensor(X, dtype=torch.float32, device=device)

model = torch.nn.Sequential(frontend, torch.nn.Linear(15, 2)).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-2)

for epoch in range(15):
    opt.zero_grad()
    logits = model(Xt)
    loss = torch.nn.functional.cross_entropy(logits, y)
    loss.backward()
    opt.step()

acc = (model(Xt).argmax(1) == y).float().mean().item()
print(f'final loss {loss.item():.4f}, train accuracy {acc:.0%}')

## Notes

- `AFX.torch.supported_features()` lists what is available on this path; everything else remains available through the NumPy API (`AFX.extract_all_features`).
- `zero_crossing_rate` and `spectral_rolloff` involve counting/selection and are not differentiable w.r.t. the waveform (gradients are zero); all other transforms propagate useful gradients.
- PyTorch stays an **optional** dependency: `import AFX` works without it.